In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import pickle
import pathlib
from pathlib import Path, PurePosixPath
import shutil

In [2]:
from pathlib import Path

#--------------------------------------------------------------------------------------------------------------------------
# pdf
#--------------------------------------------------------------------------------------------------------------------------

data_name = "MSHT20N3LO-MC-Final"
output_name = "MSHT20N3LO-MC-Final"
dataset_name = "Default"
total_xsec_name = "approximate"
if_diag_only = False
approximate_total_xsec = (total_xsec_name == "approximate")

In [3]:
#--------------------------------------------------------------------------------------------------------------------------
# resolve paths
#--------------------------------------------------------------------------------------------------------------------------

def safe_folder_name(name, label):
    name = str(name).strip()
    if not name or name in {".", ".."} or "/" in name or "\\" in name:
        raise ValueError(f"{label} must be a single folder name, got {name!r}")
    return name

data_name = safe_folder_name(data_name, "data_name")
output_name = safe_folder_name(output_name, "output_name")

cwd = Path.cwd().resolve()
data_dir_candidates = [
    cwd,
    cwd / "Data",
    cwd.parent / "Data",
]
data_root = next(
    (
        path for path in data_dir_candidates
        if (path / dataset_name).exists() and (path / "DY_total_xsec").exists()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError(
        f"Could not locate the Data directory from {cwd}. "
        "Run the notebook from TMD-Fits-Minimal or TMD-Fits-Minimal/Data."
    )

file_root = data_root / dataset_name / "Cutted" / "DY"
total_root = data_root / "DY_total_xsec" / f"{total_xsec_name}.csv"

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

pdf_matrices_root = (data_root / "PDF_Matrices").resolve()
chi2_matrix_root = (data_root / "Chi2_Matrix").resolve()
pdf_differences_root = (data_root / "PDF_Differences").resolve()

after_dir = (pdf_matrices_root / output_name).resolve()
chi2_after_dir = chi2_matrix_root
diff_input_dir = (pdf_differences_root / data_name).resolve()

if after_dir.parent != pdf_matrices_root:
    raise ValueError(f"Resolved output path escaped PDF_Matrices root: {after_dir}")
if diff_input_dir.parent != pdf_differences_root:
    raise ValueError(f"Resolved PDF-difference input path escaped PDF_Differences root: {diff_input_dir}")

print(f"Data root: {data_root}")
print(f"PDF matrix output: {after_dir}")
print(f"Chi2 matrix output: {chi2_after_dir}")
print(f"PDF difference input: {diff_input_dir}")


Data root: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data
PDF matrix output: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final
Chi2 matrix output: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix
PDF difference input: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Differences\MSHT20N3LO-MC-Final


## Dataset Selection


In [4]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]
excludes = [
    "E772/E772-5Q6.csv",
    "E772/E772-6Q7.csv",
    "E772/E772-7Q8.csv",
    "E772/E772-8Q9.csv",
]

## Input Setup


In [5]:
processes = ["DY","SIDIS"]

# Delay cleanup until after the input source has been loaded.
after_dir = Path(after_dir)
chi2_after_dir = Path(chi2_after_dir)
diff_input_dir = Path(diff_input_dir)


## Helpers


In [6]:
def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df


def reformat(d):
    out = {}
    for k, v in d.items():
        p = PurePosixPath(k)
        newk = PurePosixPath(*p.parts[-3:]).with_suffix("").as_posix()
        out[newk] = v
    return out


def get_roots(keys):
    return sorted({"/".join(PurePosixPath(k).parts[:2]) for k in keys})


def keys_for_root(keys, root):
    root = PurePosixPath(root)
    ks = [k for k in keys if PurePosixPath(k).parent == root]
    ks.sort(key=lambda x: int(PurePosixPath(x).name))
    return ks


def normalize_root_name(name):
    root = str(name).replace("\\", "/")
    if root.endswith(".csv"):
        root = root[:-4]
    return root


def expected_roots_from_files(file_root, experiments, excludes):
    excluded_roots = {normalize_root_name(root) for root in excludes}
    expected = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        if not exp_dir.exists():
            raise FileNotFoundError(f"Missing experiment directory: {exp_dir}")
        for p in sorted(exp_dir.glob("*.csv")):
            root = normalize_root_name(f"{experiment}/{p.name}")
            if root in excluded_roots:
                continue
            expected.append(root)
    return expected


def list_numeric_pickles(folder):
    if folder is None:
        return []
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(
        [p for p in folder.glob("*.pkl") if p.stem.isdigit() and int(p.stem) >= 1],
        key=lambda p: int(p.stem),
    )


## Input Discovery


In [7]:
diff_input_files = list_numeric_pickles(diff_input_dir)

if diff_input_files:
    replica_files = diff_input_files
else:
    raise RuntimeError(f"No PDF-difference replicas found under {diff_input_dir}")

N_replicas = len(replica_files)
print(N_replicas, "replicas")
print("Input mode: differences")


100 replicas
Input mode: differences


## Replica Loading


In [8]:
with open(replica_files[0], "rb") as f:
    df_1 = reformat(pickle.load(f))

keys = list(df_1.keys())
roots = get_roots(keys)
input_root_set = set(roots)
expected_roots = expected_roots_from_files(file_root, experiments, excludes)
missing_roots = [root for root in expected_roots if root not in input_root_set]
if missing_roots:
    raise RuntimeError(
        "Replica input is missing expected fit files: "
        f"{missing_roots}."
    )

extra_roots = sorted(root for root in roots if root not in set(expected_roots))
if extra_roots:
    print(f"Ignoring extra replica roots not used by the fit: {extra_roots}")

active_roots = expected_roots
print(f"Arranged {len(active_roots)} files for output and full covariance.")

replicas = []
for pkl in replica_files:
    with open(pkl, "rb") as f:
        replicas.append(reformat(pickle.load(f)))


Ignoring extra replica roots not used by the fit: ['E772/E772-5Q6', 'E772/E772-6Q7', 'E772/E772-7Q8', 'E772/E772-8Q9']
Arranged 57 files for output and full covariance.


## Optional Normalization Inputs


In [9]:
if approximate_total_xsec:
    df_total_xsec = to_float64(pd.read_csv(total_root))
    total_xsec_names = df_total_xsec["name"].tolist()
else:
    total_xsec_names = []


## Per-File Covariance Blocks


In [10]:
Matrices = {}
Centered = {}

for root in active_roots:
    ks = keys_for_root(keys, root)
    n_points = len(ks)

    X = np.empty((N_replicas, n_points), dtype=float)
    for irep, replica in enumerate(replicas):
        X[irep, :] = [replica[key] for key in ks]
    reference = np.mean(X, axis=0)
    D = X - reference
    Centered[root] = D

    if N_replicas > 1:
        Matrices[root] = (D.T @ D) / (N_replicas - 1)
    else:
        Matrices[root] = np.zeros((n_points, n_points), dtype=float)


## Full Covariance Arrangement


In [11]:
root_slices = {}
full_point_order = []
arrangement_rows = []
point_rows = []
offset = 0

for root in active_roots:
    ks = keys_for_root(keys, root)
    n_points = len(ks)
    root_slices[root] = slice(offset, offset + n_points)
    full_point_order.extend(ks)
    arrangement_rows.append({
        "file": f"{root}.csv",
        "start": offset,
        "stop": offset + n_points,
        "n_points": n_points,
    })
    for local_index, key in enumerate(ks):
        point_rows.append({
            "global_index": offset + local_index,
            "file": f"{root}.csv",
            "local_index": local_index,
            "key": key,
        })
    offset += n_points

arrangement_df = pd.DataFrame(arrangement_rows)
full_index_df = pd.DataFrame(point_rows)
N_full_points = offset
print(f"Full covariance covers {N_full_points} points.")

if active_roots:
    Full_D = np.concatenate([Centered[root] for root in active_roots], axis=1)
else:
    Full_D = np.zeros((N_replicas, 0), dtype=float)

if N_replicas > 1:
    Full_Matrix = (Full_D.T @ Full_D) / (N_replicas - 1)
else:
    Full_Matrix = np.zeros((N_full_points, N_full_points), dtype=float)


Full covariance covers 465 points.


## Write Outputs


In [12]:
after_dir = Path(after_dir)
chi2_after_dir = Path(chi2_after_dir)

if after_dir.exists():
    shutil.rmtree(after_dir)
after_dir.mkdir(parents=True, exist_ok=True)

chi2_after_dir.mkdir(parents=True, exist_ok=True)

for file_name in active_roots:
    matrix = Matrices[file_name]
    destination = after_dir / "DY" / f"{file_name}.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        raise FileExistsError(f"{destination} already exists")
    pd.DataFrame(matrix).to_csv(destination, index=False)
    print(f"{destination} generated")

full_destination = chi2_after_dir / "PDF_cov.csv"
if full_destination.exists():
    raise FileExistsError(f"{full_destination} already exists")
pd.DataFrame(Full_Matrix).to_csv(full_destination, index=False)
print(f"{full_destination} generated")

index_destination = chi2_after_dir / "Index.csv"
if index_destination.exists():
    raise FileExistsError(f"{index_destination} already exists")
full_index_df.to_csv(index_destination, index=False)
print(f"{index_destination} generated")


C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_7\ATLAS7-00y10.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_7\ATLAS7-10y20.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_7\ATLAS7-20y24.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_8\ATLAS8-00y04.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_8\ATLAS8-04y08.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_8\ATLAS8-08y12.csv generated
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\PDF_Matrices\MSHT20N3LO-MC-Final\DY\ATLAS_8\ATLAS8-116Q150.csv generated
C:\Users\congyue zhang\Desktop\O

## Full Covariance Spectrum


In [13]:
Full_Eigenvalues, Full_Eigenvectors = np.linalg.eigh(Full_Matrix)

eigen_tol = max(1.0, np.max(np.abs(Full_Eigenvalues), initial=0.0)) * 1e-12
positive_mask = Full_Eigenvalues > eigen_tol

PDF_Rep = Full_Eigenvectors[:, positive_mask] * np.sqrt(Full_Eigenvalues[positive_mask])

n_total_eigenvectors = len(Full_Eigenvalues)
n_nonzero_eigenvalues = int(np.count_nonzero(np.abs(Full_Eigenvalues) > eigen_tol))
n_positive_eigenvalues = int(np.count_nonzero(positive_mask))
n_negative_eigenvalues = int(np.count_nonzero(Full_Eigenvalues < -eigen_tol))

print(f"Total eigenvectors: {n_total_eigenvectors}")
print(f"Numerically nonzero eigenvalues: {n_nonzero_eigenvalues}")
print(f"Positive eigenvalues retained: {n_positive_eigenvalues}")
print(f"Negative eigenvalues beyond tolerance: {n_negative_eigenvalues}")
print(f"Eigenvalue tolerance: {eigen_tol:.3e}")

mode_labels = [f"mode_{i + 1}" for i in range(n_positive_eigenvalues)]

pdf_rep_destination = chi2_after_dir / "PDF_rep.csv"
if pdf_rep_destination.exists():
    raise FileExistsError(f"{pdf_rep_destination} already exists")
pd.DataFrame(PDF_Rep, columns=mode_labels).to_csv(pdf_rep_destination, index=False)
print(f"{pdf_rep_destination} generated")


Total eigenvectors: 465
Numerically nonzero eigenvalues: 99
Positive eigenvalues retained: 99
Negative eigenvalues beyond tolerance: 0
Eigenvalue tolerance: 1.066e-11
C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix\PDF_rep.csv generated
